<a href="https://colab.research.google.com/github/EliasNoorzad/spacecraft-pose-estimation/blob/main/notebooks/yolo_test_evaluation_10k.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# DLVS3-HST — YOLOv8s-Pose test evaluation (10K)

Evaluate the trained `best.pt` checkpoint on the **1,000 held-out test images**. This notebook reports standard YOLO box/pose metrics and 2D keypoint errors in pixels. No training or PnP is performed.

The processed dataset's YOLO labels provide the ground-truth 2D keypoints. If the dataset is still in Colab's `/content` storage from training, no new download is needed.

In [1]:
!pip -q install ultralytics pyyaml pandas

## 1. Load the saved model and test dataset

In [2]:
from google.colab import drive
from pathlib import Path
from zipfile import ZipFile
import shutil
import torch

drive.mount('/content/drive')

DATASET_ROOT = Path('/content/hst_yolo_pose_10k')
DRIVE_ZIP = Path('/content/drive/MyDrive/HST/hst_yolo_pose_10k.zip')
RUN_DIR = Path('/content/drive/MyDrive/HST/HST_YOLO_Runs/hst_pose_10k_sigma_010')
BEST_PT = RUN_DIR / 'weights/best.pt'

# Reuse the local dataset when running after training.
if not (DATASET_ROOT / 'images/test').is_dir():
    if DATASET_ROOT.exists():
        raise RuntimeError('Incomplete local dataset at /content/hst_yolo_pose_10k. Start a fresh runtime.')
    local_zip = Path('/content/hst_yolo_pose_10k.zip')
    shutil.copyfile(DRIVE_ZIP, local_zip)
    with ZipFile(local_zip) as archive:
        archive.extractall('/content')
    local_zip.unlink()

TEST_IMAGES = DATASET_ROOT / 'images/test'
TEST_LABELS = DATASET_ROOT / 'labels/test'
assert TEST_IMAGES.is_dir() and TEST_LABELS.is_dir(), 'Test images or labels are missing.'
assert BEST_PT.is_file(), f'Checkpoint not found: {BEST_PT}'
assert torch.cuda.is_available(), 'Select a GPU runtime in Colab.'

print('GPU:', torch.cuda.get_device_name(0))
print('Test images:', TEST_IMAGES)
print('Checkpoint:', BEST_PT)

Mounted at /content/drive
GPU: NVIDIA L4
Test images: /content/hst_yolo_pose_10k/images/test
Checkpoint: /content/drive/MyDrive/HST/HST_YOLO_Runs/hst_pose_10k_sigma_010/weights/best.pt


## 2. Evaluate the held-out test split

In [3]:
import yaml
from ultralytics import YOLO

# Keep the custom OKS sigma used during training.
with (DATASET_ROOT / 'data.yaml').open() as f:
    config = yaml.safe_load(f)
config['path'] = str(DATASET_ROOT)
config['kpt_shape'] = [37, 3]
config['kpt_oks_sigmas'] = [0.10] * 37
TEST_YAML = Path('/content/hst_10k_test.yaml')
with TEST_YAML.open('w') as f:
    yaml.safe_dump(config, f, sort_keys=False)

model = YOLO(str(BEST_PT))
metrics = model.val(
    data=str(TEST_YAML),
    split='test',
    imgsz=640,
    batch=8,
    device=0,
    workers=2,
    verbose=True,
    plots=True,
    project=str(RUN_DIR),
    name='test_evaluation',
)

print('\nTEST RESULTS (1,000 held-out images)')
print(f"{'':19s}{'Precision':>10s}{'Recall':>10s}{'mAP50':>10s}{'mAP50-95':>12s}")
for name, m in [('Box', metrics.box), ('Pose', metrics.pose)]:
    print(f'{name:19s}{m.mp:10.3f}{m.mr:10.3f}{m.map50:10.3f}{m.map:12.3f}')

Creating new Ultralytics Settings v0.0.8 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/usage/settings
Ultralytics 8.4.216 🚀 Python-3.13.16 torch-2.14.1+cu130 CUDA:0 (NVIDIA L4, 22563MiB)
YOLOv8s-pose summary (fused): 81 layers, 12,391,704 parameters, 0 gradients, 33.4 GFLOPs
val: Fast image access ✅ (ping: 0.0±0.0 ms, read: 3879.8±1045.0 MB/s, size: 662.0 KB)
val: Scanning /content/hst_yolo_pose_10k/labels/test... 1000 images, 0 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 1000/1000 229.6it/s 4.4s
val: New cache created: /content/hst_yolo_pose_10k/labels/test.cache
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95)     Pose(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 125/125 12.1it/s 10.4s
                   all       1000       1000      0.999          1

## 3. Measure 2D keypoint localization errors

As in our smoke test, use the first detected object from each image (`conf=0.25`). Compare predicted pixel positions to in-frame keypoints in the matching YOLO label. Invalid/out-of-frame ground-truth keypoints (visibility `0`) are excluded. Image files are matched with labels by filename, including any names changed during preprocessing.

In [4]:
import numpy as np
import pandas as pd
from IPython.display import display

records = []
missing_detections = 0
processed = 0

predictions = model.predict(
    source=str(TEST_IMAGES),
    imgsz=640,
    batch=8,
    conf=0.25,
    device=0,
    save=False,
    verbose=False,
    stream=True,
)

for result in predictions:
    processed += 1
    image_path = Path(result.path)
    label_path = TEST_LABELS / (image_path.stem + '.txt')

    # The converted dataset contains one HST annotation per image.
    values = np.fromstring(label_path.read_text().strip(), sep=' ')
    if values.size != 5 + 37 * 3:
        raise ValueError(f'Unexpected YOLO pose label: {label_path}')

    if result.keypoints is None or len(result.keypoints.xy) == 0:
        missing_detections += 1
        continue

    h, w = result.orig_shape
    gt = values[5:].reshape(37, 3)
    gt_xy = gt[:, :2] * np.array([w, h], dtype=np.float64)
    valid = (gt[:, 2] > 0) & np.isfinite(gt_xy).all(axis=1)

    predicted_xy = result.keypoints.xy[0].cpu().numpy()
    if predicted_xy.shape != (37, 2):
        raise ValueError(f'Unexpected prediction shape for {image_path.name}: {predicted_xy.shape}')

    for i in np.flatnonzero(valid):
        error = float(np.linalg.norm(predicted_xy[i] - gt_xy[i]))
        records.append({
            'Image': image_path.name,
            'Keypoint': i + 1,
            'Pixel Error': error,
            'Zero Prediction': bool(np.all(predicted_xy[i] == 0)),
        })

df = pd.DataFrame(records)
if df.empty:
    raise RuntimeError('No keypoints were evaluated.')

errors = df['Pixel Error']
print('\n========== YOLO KEYPOINT ACCURACY ==========')
print(f'Test images processed:  {processed}')
print(f'Images evaluated:       {df["Image"].nunique()}')
print(f'Missing detections:     {missing_detections}')
print(f'Keypoints evaluated:    {len(df)}')
print(f'Zero predictions:       {df["Zero Prediction"].sum()}')
print(f'\nMean pixel error:       {errors.mean():.3f} px')
print(f'Median pixel error:     {errors.median():.3f} px')
print(f'90th percentile error:  {errors.quantile(0.90):.3f} px')
print(f'Maximum pixel error:    {errors.max():.3f} px')
print(f'\nWithin 5 pixels:        {(errors <= 5).mean() * 100:.2f}%')
print(f'Within 10 pixels:       {(errors <= 10).mean() * 100:.2f}%')
print(f'Within 20 pixels:       {(errors <= 20).mean() * 100:.2f}%')

print('\n========== ERROR BY KEYPOINT ==========')
display(df.groupby('Keypoint')['Pixel Error'].agg(['mean', 'median', 'max']).round(3))

print('\n========== 10 WORST IMAGES ==========')
display(df.groupby('Image')['Pixel Error'].agg(['mean', 'median', 'max'])
        .sort_values('mean', ascending=False).head(10).round(3))


========== YOLO KEYPOINT ACCURACY ==========
Test images processed:  1000
Images evaluated:       1000
Missing detections:     0
Keypoints evaluated:    36985
Zero predictions:       0

Mean pixel error:       6.382 px
Median pixel error:     4.125 px
90th percentile error:  10.918 px
Maximum pixel error:    419.013 px

Within 5 pixels:        60.16%
Within 10 pixels:       87.81%
Within 20 pixels:       97.14%

========== ERROR BY KEYPOINT ==========


,mean,median,max
Keypoint,,,
1,8.411,5.837,419.013
2,6.306,4.306,266.648
3,6.250,4.327,326.209
4,5.893,4.113,172.042
5,5.942,4.082,279.882
6,4.047,2.958,79.167
7,4.271,3.180,76.745
8,9.056,5.883,236.826
9,9.257,6.144,250.558



========== 10 WORST IMAGES ==========


,mean,median,max
Image,,,
output_0000958.png,161.667,152.031,419.013
output_0003683.png,143.876,114.934,310.655
output_0001832.png,114.825,84.019,224.601
output_0009017.png,112.894,100.683,241.939
output_0000993.png,103.930,82.467,212.340
output_0000887.png,79.379,72.945,226.758
output_0004092.png,77.818,69.014,221.468
output_0002579.png,76.173,64.480,167.381
output_0001924.png,72.567,52.655,251.931
